# Projeto A3 — Predição de Burnout em Estudantes com Inteligência Artificial

Neste notebook, foi desenvolvido um modelo de Machine Learning com o objetivo de analisar dados de estudantes e prever possíveis sinais de Burnout.

O código foi organizado em etapas para facilitar a compreensão durante a apresentação. Cada parte possui uma explicação sobre sua função dentro do projeto, mostrando desde o carregamento dos dados até a conclusão final dos comitês de modelos.


# Projeto A3 - Comitê de Classificadores para Previsão de Burnout Estudantil

**Disciplina:** Inteligência Artificial  
**Objetivo:** construir um pipeline completo de classificação e comparar modelos individuais com um comitê por votação majoritária.

## Problema de inferência

Desejamos prever o **nível de burnout estudantil** a partir de variáveis acadêmicas, psicológicas e de estilo de vida. A previsão é uma tarefa de classificação porque o resultado pertence a categorias, como níveis baixo, médio ou alto.

> Observação ética: o modelo é educacional e exploratório. Ele não substitui avaliação profissional nem deve ser usado para diagnosticar pessoas.

## 1. Instalação e importações

Execute esta célula primeiro.

## Importação das bibliotecas

Nesta parte do código, foram importadas as bibliotecas necessárias para o desenvolvimento do projeto.

Essas bibliotecas permitem realizar a leitura e organização dos dados, gerar gráficos, dividir os dados em treino e teste, treinar os modelos de Inteligência Artificial e avaliar os resultados obtidos.

Ou seja, antes de iniciar a análise, o código prepara todas as ferramentas que serão utilizadas nas próximas etapas.


In [ ]:
!pip -q install kagglehub

from pathlib import Path
import json
import re
import unicodedata
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
MAX_ROWS = 30000  # amostra estratificada para a execução continuar leve na apresentação
print("Bibliotecas carregadas com sucesso!")

## 2. Carregamento da base do Kaggle

A célula tenta baixar automaticamente a base pública. Caso isso falhe, selecione manualmente o arquivo CSV baixado do Kaggle.

## Carregamento do dataset

Nesta parte do código, foi carregado o arquivo contendo os dados dos estudantes.

O dataset é a base principal do projeto, pois contém as informações que serão analisadas pelos modelos de Inteligência Artificial. A partir desses dados, o sistema aprende padrões relacionados ao Burnout e depois tenta realizar previsões.


In [ ]:
DATASET_SLUG = "sehaj1104/student-mental-health-and-burnout-dataset"

def carregar_csv():
    try:
        import kagglehub
        pasta = Path(kagglehub.dataset_download(DATASET_SLUG))
        arquivos = list(pasta.rglob("*.csv"))
        if not arquivos:
            raise FileNotFoundError("Nenhum CSV encontrado na pasta baixada.")
        print("CSV encontrado automaticamente:", arquivos[0])
        return pd.read_csv(arquivos[0])
    except Exception as erro:
        print("Download automático não funcionou:", erro)
        print("Selecione manualmente o CSV baixado do Kaggle.")
        from google.colab import files
        enviados = files.upload()
        nome = next(iter(enviados))
        return pd.read_csv(nome)

df_original = carregar_csv()
print("Formato original:", df_original.shape)
df_original.head()

## 3. Padronização dos nomes e definição do target

O target esperado é o nível de burnout. O código também detecta nomes equivalentes automaticamente.

## Visualização inicial dos dados

Nesta parte do código, são exibidas as primeiras linhas do dataset.

Essa etapa serve para verificar se o arquivo foi carregado corretamente e para entender como as informações estão organizadas na tabela, como os nomes das colunas e os tipos de dados presentes.


In [ ]:
def normalizar_nome_coluna(nome):
    texto = unicodedata.normalize("NFKD", str(nome)).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-zA-Z0-9]+", "_", texto.strip().lower())
    return texto.strip("_")

def encontrar_target(colunas):
    preferidas = [
        "burnout_level", "burnout_category", "burnout_risk_level",
        "burnout_risk", "burnout_class", "burnout_status"
    ]
    for candidata in preferidas:
        if candidata in colunas:
            return candidata
    alternativas = [
        coluna for coluna in colunas
        if "burnout" in coluna and not any(item in coluna for item in ["score", "index", "rate"])
    ]
    if alternativas:
        return alternativas[0]
    raise ValueError("Não foi possível detectar a coluna target automaticamente.")

df = df_original.copy()
df.columns = [normalizar_nome_coluna(coluna) for coluna in df.columns]
target = encontrar_target(df.columns)
df = df.dropna(subset=[target]).reset_index(drop=True)

print("Target identificado:", target)
print("Classes:", sorted(df[target].astype(str).unique().tolist()))
df.head()

## 4. Prevenção de vazamento de dados

Identificadores não ajudam a generalizar. Além disso, uma coluna como `burnout_score` pode revelar diretamente a resposta usada para gerar o nível de burnout. Por isso, essas colunas não entram como features.

## Definição da variável principal do projeto

Nesta parte do código, é definida a variável que representa o resultado que o modelo deverá prever.

No caso deste projeto, o foco é identificar se um estudante apresenta ou não sinais de Burnout. Essa variável é chamada de variável-alvo, pois é a resposta que os modelos de Machine Learning tentarão aprender a prever.


In [ ]:
colunas_removidas = []
for coluna in df.columns:
    if coluna == target:
        continue
    identificador = coluna in {"id", "student_id", "record_id", "studentid"} or coluna.endswith("_id")
    pontuacao_direta = "burnout" in coluna and any(item in coluna for item in ["score", "index", "rate"])
    if identificador or pontuacao_direta:
        colunas_removidas.append(coluna)

df = df.drop(columns=colunas_removidas)
print("Colunas removidas:", colunas_removidas)
print("Formato após remoções:", df.shape)

## 5. Análise exploratória básica (EDA)

## Visualização inicial dos dados

Nesta parte do código, são exibidas as primeiras linhas do dataset.

Essa etapa serve para verificar se o arquivo foi carregado corretamente e para entender como as informações estão organizadas na tabela, como os nomes das colunas e os tipos de dados presentes.


In [ ]:
print("Quantidade de linhas e colunas:", df.shape)
print("\nTipos das colunas:")
display(df.dtypes.to_frame("tipo"))

print("\nValores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False).to_frame("ausentes"))

print("\nDistribuição das classes:")
display(df[target].value_counts(dropna=False).to_frame("quantidade"))

plt.figure(figsize=(7, 4))
df[target].astype(str).value_counts().sort_index().plot(kind="bar")
plt.title("Distribuição das classes do target")
plt.xlabel(target)
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()

In [ ]:
colunas_numericas = df.select_dtypes(include=np.number).columns.tolist()[:6]
if colunas_numericas:
    df[colunas_numericas].hist(figsize=(12, 8), bins=20)
    plt.suptitle("Distribuições das primeiras variáveis numéricas")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma variável numérica encontrada.")

## 6. Separação entre treino e teste

Para manter uma execução rápida durante a apresentação, usamos no máximo 30 mil registros, preservando a proporção das classes por amostragem estratificada.

## Definição da variável principal do projeto

Nesta parte do código, é definida a variável que representa o resultado que o modelo deverá prever.

No caso deste projeto, o foco é identificar se um estudante apresenta ou não sinais de Burnout. Essa variável é chamada de variável-alvo, pois é a resposta que os modelos de Machine Learning tentarão aprender a prever.


In [ ]:
if len(df) > MAX_ROWS:
    df_modelagem, _ = train_test_split(
        df,
        train_size=MAX_ROWS,
        random_state=RANDOM_STATE,
        stratify=df[target],
    )
    df_modelagem = df_modelagem.reset_index(drop=True)
else:
    df_modelagem = df.copy()

X = df_modelagem.drop(columns=[target])
y = df_modelagem[target].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Registros usados:", len(df_modelagem))
print("Treino:", len(X_train))
print("Teste:", len(X_test))

## 7. Pré-processamento

- Valores ausentes numéricos: mediana.
- Valores ausentes categóricos: categoria mais frequente.
- Variáveis numéricas: padronização.
- Variáveis categóricas: codificação One-Hot Encoding.

## Pré-processamento dos dados

Nesta parte do código, os dados foram preparados para serem interpretados pelos algoritmos de Machine Learning.

Como os modelos trabalham melhor com valores numéricos, algumas informações precisam ser convertidas ou ajustadas. Além disso, algumas técnicas ajudam a padronizar os dados para que todos tenham uma escala adequada.

Essa etapa melhora a qualidade do treinamento e evita problemas durante a execução dos modelos.


In [ ]:
def criar_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def criar_preprocessador(X_base):
    numericas = X_base.select_dtypes(include=np.number).columns.tolist()
    categoricas = X_base.select_dtypes(exclude=np.number).columns.tolist()

    pipeline_numerico = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    pipeline_categorico = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", criar_one_hot_encoder()),
    ])
    return ColumnTransformer(transformers=[
        ("num", pipeline_numerico, numericas),
        ("cat", pipeline_categorico, categoricas),
    ])

def criar_pipeline(classificador):
    return Pipeline(steps=[
        ("preprocessamento", criar_preprocessador(X_train)),
        ("classificador", classificador),
    ])

## 8. Modelos individuais

Treinaremos três algoritmos diferentes apresentados no enunciado: Árvore de Decisão, Naive Bayes e KNN.

## Pré-processamento dos dados

Nesta parte do código, os dados foram preparados para serem interpretados pelos algoritmos de Machine Learning.

Como os modelos trabalham melhor com valores numéricos, algumas informações precisam ser convertidas ou ajustadas. Além disso, algumas técnicas ajudam a padronizar os dados para que todos tenham uma escala adequada.

Essa etapa melhora a qualidade do treinamento e evita problemas durante a execução dos modelos.


In [ ]:
modelos = {
    "Árvore de Decisão": criar_pipeline(DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE)),
    "Naive Bayes": criar_pipeline(GaussianNB()),
    "KNN": criar_pipeline(KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1)),
}

def avaliar_modelo(nome, modelo):
    previsoes = modelo.predict(X_test)
    metricas = {
        "modelo": nome,
        "acuracia": accuracy_score(y_test, previsoes),
        "precisao_weighted": precision_score(y_test, previsoes, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_test, previsoes, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_test, previsoes, average="weighted", zero_division=0),
    }
    print(f"\n===== {nome.upper()} =====")
    display(pd.DataFrame([metricas]))
    print(classification_report(y_test, previsoes, zero_division=0))

    rotulos = sorted(y_test.astype(str).unique().tolist())
    matriz = confusion_matrix(y_test.astype(str), pd.Series(previsoes).astype(str), labels=rotulos)
    ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=rotulos).plot(cmap="Blues", xticks_rotation=45)
    plt.title(f"Matriz de confusão - {nome}")
    plt.tight_layout()
    plt.show()
    return metricas

resultados = []
for nome, modelo in modelos.items():
    print("Treinando:", nome)
    modelo.fit(X_train, y_train)
    resultados.append(avaliar_modelo(nome, modelo))

## 9. Comitê de classificadores

Utilizamos **votação majoritária** (`voting="hard"`). Cada classificador possui um voto e a categoria mais votada é escolhida como previsão final. Essa estratégia é simples, interpretável e combina modelos com comportamentos diferentes.

## Treinamento do modelo Árvore de Decisão

Nesta parte do código, foi treinado o modelo chamado Árvore de Decisão.

Esse modelo funciona de forma semelhante a um fluxograma, criando regras a partir dos dados. Ele analisa as características dos estudantes e toma decisões passo a passo até chegar a uma previsão sobre Burnout.


In [ ]:
comite = VotingClassifier(
    estimators=[
        ("arvore", modelos["Árvore de Decisão"]),
        ("naive_bayes", modelos["Naive Bayes"]),
        ("knn", modelos["KNN"]),
    ],
    voting="hard",
)

print("Treinando comitê...")
comite.fit(X_train, y_train)
resultados.append(avaliar_modelo("Comitê - Votação Majoritária", comite))

## 10. Comparação final

In [ ]:
comparacao = pd.DataFrame(resultados).sort_values("f1_weighted", ascending=False).reset_index(drop=True)
display(comparacao)

melhor = comparacao.iloc[0]
print(f"Melhor resultado por F1-score ponderado: {melhor['modelo']} ({melhor['f1_weighted']:.4f})")
comparacao.to_csv("comparacao_modelos.csv", index=False)
print("Arquivo comparacao_modelos.csv salvo com sucesso.")

## 11. Guia para a conclusão do relatório

Após executar o notebook, preencham a conclusão com os valores exibidos na tabela final:

1. Informem qual classificador individual apresentou o maior F1-score.
2. Compare esse valor com o resultado do comitê.
3. Digam se o comitê melhorou, empatou ou piorou o desempenho.
4. Comentem que a base é sintética e que o modelo é exploratório, não uma ferramenta de diagnóstico.
5. Como melhoria futura, sugiram ajuste de hiperparâmetros e validação cruzada.

## 12. Conclusão pronta para o relatório

Neste projeto, foi desenvolvido um pipeline de classificação para estimar o nível de burnout estudantil a partir de variáveis acadêmicas, psicológicas e de estilo de vida. Foram avaliados três classificadores individuais — Árvore de Decisão, Naive Bayes e KNN — além de um comitê de classificadores com votação majoritária.

Considerando o **F1-score ponderado**, o melhor desempenho individual foi obtido pelo **KNN**, com valor de **0,3279**. O comitê de classificadores apresentou F1-score ponderado de **0,3177**. Portanto, neste experimento, o comitê **piorou levemente o desempenho** em relação ao melhor modelo individual, com uma diferença aproximada de **0,0102** no F1-score, equivalente a cerca de **1,02 ponto percentual**.

Os resultados ficaram próximos de 0,33, o que indica dificuldade dos modelos em distinguir claramente as três classes de burnout presentes na base. Como as classes possuem distribuição semelhante, esse resultado também sugere que as variáveis disponíveis não permitiram uma separação forte entre os níveis `Low`, `Medium` e `High` com os modelos e configurações utilizados.

É importante destacar que a base utilizada é **sintética**. Dessa forma, este projeto deve ser interpretado como um estudo exploratório de aplicação de técnicas de Inteligência Artificial, e não como uma ferramenta de diagnóstico de saúde mental. Qualquer aplicação real exigiria dados reais, validação especializada e cuidados éticos adicionais.

Como melhorias futuras, recomenda-se realizar ajuste de hiperparâmetros, aplicar validação cruzada e testar outras estratégias de ensemble, como votação ponderada. Também seria interessante avaliar novos modelos e investigar quais variáveis possuem maior influência nas previsões.


## 13. Teste final com 1 usuário e conclusão dos comitês

Nesta etapa final, o notebook seleciona **1 usuário do conjunto de teste**, passa esse usuário por cada classificador e mostra se cada comitê/modelo indicou **Sim** ou **Não** para burnout.

Regra usada:

- **Low / Baixo / Sem Burnout** = **Não possui burnout**
- **Medium / High / Médio / Alto** = **Possui burnout**


## Treinamento do modelo Árvore de Decisão

Nesta parte do código, foi treinado o modelo chamado Árvore de Decisão.

Esse modelo funciona de forma semelhante a um fluxograma, criando regras a partir dos dados. Ele analisa as características dos estudantes e toma decisões passo a passo até chegar a uma previsão sobre Burnout.


In [ ]:
# ==========================================
# 13. Teste final com 1 usuário
# ==========================================

# Seleciona 1 usuário do conjunto de teste
usuario_teste = X_test.iloc[[0]]
resultado_real = y_test.iloc[0]

print("===== USUÁRIO TESTADO =====")
display(usuario_teste)

print("Resultado real do dataset:", resultado_real)


def burnout_sim_nao(resultado):
    """Converte a classe prevista em Sim/Não para facilitar a apresentação."""
    texto = str(resultado).strip().lower()

    classes_sem_burnout = [
        "low", "baixo", "baixa", "sem burnout",
        "no", "nao", "não", "none", "normal"
    ]

    if texto in classes_sem_burnout:
        return "Não"
    return "Sim"


# Comitês/modelos que serão exibidos no resultado final
avaliacoes_finais = {
    "Comitê Árvore de Decisão": modelos["Árvore de Decisão"],
    "Comitê Naive Bayes": modelos["Naive Bayes"],
    "Comitê KNN": modelos["KNN"],
    "Comitê Final - Votação Majoritária": comite
}

resultados_usuario = []

for nome_comite, modelo in avaliacoes_finais.items():
    previsao = modelo.predict(usuario_teste)[0]
    possui_burnout = burnout_sim_nao(previsao)

    resultados_usuario.append({
        "Comitê": nome_comite,
        "Resultado previsto": previsao,
        "Possui burnout?": possui_burnout
    })

tabela_resultado_usuario = pd.DataFrame(resultados_usuario)

print("\n===== RESULTADO FINAL DOS COMITÊS PARA 1 USUÁRIO =====")
display(tabela_resultado_usuario)

print("\n===== RESUMO PARA APRESENTAÇÃO =====")
for item in resultados_usuario:
    print(f"{item['Comitê']} ---- {item['Possui burnout?']}")

resultado_comite_final = tabela_resultado_usuario.loc[
    tabela_resultado_usuario["Comitê"] == "Comitê Final - Votação Majoritária",
    "Possui burnout?"
].iloc[0]

print("\nConclusão final:")
if resultado_comite_final == "Sim":
    print("De acordo com o Comitê Final por Votação Majoritária, este usuário POSSUI indicativo de burnout.")
else:
    print("De acordo com o Comitê Final por Votação Majoritária, este usuário NÃO POSSUI indicativo de burnout.")


## 14. Conclusão final entre os comitês

Com o teste de um usuário individual, foi possível comparar a decisão de cada comitê/modelo. Cada classificador apresentou sua própria previsão e, ao final, o **Comitê Final por Votação Majoritária** reuniu essas decisões para gerar uma conclusão única.

Esse resultado final permite demonstrar, de forma simples, como diferentes modelos podem chegar a respostas diferentes e como o comitê ajuda a consolidar uma decisão final para indicar se o usuário possui ou não indicativo de burnout.


# Conclusão final do projeto

Nesta etapa final, é possível observar que o projeto utilizou diferentes modelos de Machine Learning para analisar dados de estudantes e prever possíveis sinais de Burnout.

Foram utilizados modelos individuais, como Árvore de Decisão, KNN e Naive Bayes, além de um Comitê de Votação, que combina as previsões dos modelos para gerar uma decisão final.

A proposta do projeto foi demonstrar como a Inteligência Artificial pode auxiliar na análise de dados relacionados à saúde mental dos estudantes, identificando padrões que podem indicar situações de risco.

Dessa forma, o sistema não substitui uma avaliação profissional, mas pode servir como uma ferramenta de apoio para ajudar na identificação precoce de possíveis casos de Burnout.
